# PDFs and CDFs in data analysis
Class PDFs for feature selection, CDFs to measure a rule, and a 2D density plot, on the iris data.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from sklearn.datasets import load_iris

# iris from scikit-learn: works offline
data = load_iris(as_frame=True)
iris = data.frame.rename(columns=lambda c: c.replace(" (cm)", "").replace(" ", "_"))
iris["species"] = iris["target"].map(dict(enumerate(data.target_names)))
iris.head()

## 1. One PDF (KDE) per species for each feature

In [ ]:
features = ["petal_length", "petal_width", "sepal_length", "sepal_width"]
fig = make_subplots(rows=2, cols=2, subplot_titles=features)
grid = np.linspace(0, 8.5, 400)
for i, feat in enumerate(features):
    for name in data.target_names:
        values = iris.loc[iris["species"] == name, feat]
        fig.add_scatter(x=grid, y=stats.gaussian_kde(values)(grid), name=name, legendgroup=name,
                        showlegend=(i == 0), row=i // 2 + 1, col=i % 2 + 1)
fig.update_layout(template="simple_white", height=600)
fig.show()

## 2. A rule from petal length, checked with the data

In [ ]:
pl = iris["petal_length"]
for name in data.target_names:
    v = pl[iris["species"] == name]
    print(f"{name:10s} below 2.3: {(v < 2.3).mean():.2f}   at most 5: {(v <= 5).mean():.2f}")

## 3. The empirical CDF of petal width at 1.7

In [ ]:
pw = iris["petal_width"]
fig = go.Figure()
for name in data.target_names:
    v = np.sort(pw[iris["species"] == name])
    ecdf = np.arange(1, len(v) + 1) / len(v)      # share at or below each value
    fig.add_scatter(x=v, y=ecdf, mode="lines", line_shape="hv", name=name)
    print(f"{name:10s} share at or below 0.7: {(v <= 0.7).mean():.2f}, at or below 1.7: {(v <= 1.7).mean():.2f}")
fig.add_vline(x=0.7, line_dash="dash")
fig.add_vline(x=1.7, line_dash="dash")
fig.update_layout(template="simple_white", xaxis_title="petal width", yaxis_title="share <= x")
fig.show()

In [ ]:
# accuracy of the whole rule: <= 0.7 setosa, (0.7, 1.7] versicolor, > 1.7 virginica
pred = np.where(pw <= 0.7, "setosa", np.where(pw <= 1.7, "versicolor", "virginica"))
print("correct:", (pred == iris["species"]).sum(), "of", len(iris))
pd.crosstab(iris["species"], pred)

## 4. A 2D density plot

In [ ]:
x, y = iris["petal_length"], iris["sepal_length"]
gx, gy = np.linspace(0, 8, 120), np.linspace(3.5, 8.5, 120)
X, Y = np.meshgrid(gx, gy)
# 2D KDE evaluated on a grid
Z = stats.gaussian_kde(np.vstack([x, y]))(np.vstack([X.ravel(), Y.ravel()])).reshape(X.shape)
fig = go.Figure(go.Contour(x=gx, y=gy, z=Z, colorscale="Blues"))
fig.add_scatter(x=x, y=y, mode="markers", marker=dict(size=4, color="orange"))
fig.update_layout(template="simple_white", xaxis_title="petal length", yaxis_title="sepal length")
fig.show()